# TEMPLATE 3: HIGH-CARDINALITY MULTI-CLASS TEXT CLASSIFICATION
### GEMASTIK DATA MINING & NLP PIPELINE (KOMPAS 28 KELAS & KAMPUS 11 BIRO)
Pipeline ini dirancang untuk menyelesaikan kompetisi klasifikasi teks dengan jumlah kelas banyak (10-30 kelas) dan ketimpangan data ekstrem (*class imbalance*).

**Fitur Utama Pipeline:**
1. Modular Cleaning & Field Fusion (`[TITLE] ... [CONTENT] ...`).
2. Rare Class Shielding: mencegah error *train_test_split* pada kelas yang beranggotakan hanya 1 sampel.
3. Model 1: Sublinear TF-IDF + Balanced Multinomial Logistic Regression (Cepat di CPU).
4. Model 2: IndoBERT Fine-Tuning dengan Custom WeightedTrainer (Weighted CrossEntropyLoss) untuk akselerasi GPU RTX 3070.
5. Laporan Detail Presisi, Recall, dan Macro-F1 per kategori.
6. Pembuatan berkas submission multi-kelas.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

SEED = 2026
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device aktif: {device} | {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'}")


## 1. Memuat Dataset Kompetisi
Arahkan ke berkas dataset Anda (misal `cases/nlp_kompas/train.csv` atau `cases/campus_complaints/train.csv`).


In [ ]:
TRAIN_PATH = "cases/nlp_kompas/train.csv"
TEXT_COL = "clean_text"
TARGET_COL = "category"

if not os.path.exists(TRAIN_PATH):
    print("Membuat sampel data multikelas...")
    sample_data = {
        "id": range(1, 16),
        "clean_text": [
            "pemain timnas sepak bola mencetak gol di babak kedua piala asia",
            "pertandingan bola liga inggris berlangsung sengit tadi malam",
            "ihsg ditutup menguat didorong saham perbankan dan emiten tambang",
            "bank indonesia mempertahankan suku bunga acuan untuk stabilkan inflasi",
            "smartphone terbaru ini dibekali chipset kecerdasan buatan bertenaga ai",
            "perusahaan teknologi merilis sistem operasi baru untuk laptop",
            "resep membuat masakan ayam rendang khas minang yang empuk dan lezat",
            "kuliner kopi lokal kini makin digemari kalangan muda di berbagai kafe",
            "peneliti lapan mengamati fenomena hujan meteor di langit malam",
            "teleskop luar angkasa menemukan planet baru berpotensi layak huni",
            "vaksinasi gratis dibuka untuk masyarakat umum di puskesmas",
            "tips menjaga kesehatan jantung dengan rajin berolahraga kardio",
            "wisata pantai di pesisir selatan menyajikan pemandangan sunset indah",
            "penerbangan internasional ke destinasi wisata bali mulai meningkat",
            "kementerian pendidikan meluncurkan beasiswa kuliah untuk mahasiswa berprestasi"
        ],
        "category": [
            "Bola", "Bola", "Money", "Money", "Tekno", "Tekno", "Food", "Food",
            "Sains", "Sains", "Health", "Health", "Travel", "Travel", "Edukasi"
        ]
    }
    df = pd.DataFrame(sample_data)
else:
    df = pd.read_csv(TRAIN_PATH)

df = df.dropna(subset=[TEXT_COL, TARGET_COL]).reset_index(drop=True)
categories = sorted(df[TARGET_COL].unique())
num_classes = len(categories)
label2id = {c: i for i, c in enumerate(categories)}
id2label = {i: c for i, c in enumerate(categories)}
df["label"] = df[TARGET_COL].map(label2id)

print(f"Total Dokumen: {len(df)} | Total Kategori Target: {num_classes}")
print(df[TARGET_COL].value_counts().head(10))


## 2. Rare Class Shielding (Proteksi Kelas Langka < 2 Sampel)


In [ ]:
class_counts = df["label"].value_counts()
rare_classes = class_counts[class_counts < 2].index.tolist()

if rare_classes:
    rare_names = [id2label[rc] for rc in rare_classes]
    print(f"Mengisolasi kelas langka (< 2 sampel) ke data latih: {rare_names}")
    rare_df = df[df["label"].isin(rare_classes)]
    strat_df = df[~df["label"].isin(rare_classes)].reset_index(drop=True)
    train_sub, val_df = train_test_split(strat_df, test_size=0.2, random_state=SEED, stratify=strat_df["label"])
    train_df = pd.concat([train_sub, rare_df], axis=0).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
else:
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["label"])

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"Data Latih: {len(train_df)} | Data Validasi: {len(val_df)}")


## 3. Model 1: Sublinear TF-IDF + Balanced Logistic Regression (Baseline Cepat)


In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=25000, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL])
X_val_tfidf = tfidf.transform(val_df[TEXT_COL])

lr_model = LogisticRegression(C=1.0, max_iter=500, class_weight="balanced", random_state=SEED)
lr_model.fit(X_train_tfidf, train_df["label"])

lr_preds = lr_model.predict(X_val_tfidf)
print("=== HASIL BASELINE TF-IDF + LOGISTIC REGRESSION ===")
print(f"Akurasi  : {accuracy_score(val_df['label'], lr_preds) * 100:.2f}%")
print(f"Macro-F1 : {f1_score(val_df['label'], lr_preds, average='macro', zero_division=0) * 100:.2f}%")


## 4. Model 2: IndoBERT Fine-Tuning dengan Weighted Loss (Akselerasi RTX 3070)


In [ ]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# Hitung bobot kelas untuk Weighted Loss
raw_weights = compute_class_weight(class_weight="balanced", classes=np.arange(num_classes), y=train_df["label"].values)
clamped_weights = np.clip(raw_weights, a_min=0.2, a_max=30.0)
weights_tensor = torch.tensor(clamped_weights, dtype=torch.float).to(device)

MODEL_DIR = "./models/indobert-base-p1"
if not os.path.exists(MODEL_DIR):
    MODEL_DIR = "indobenchmark/indobert-base-p1"

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    
    class TextClassificationDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_len=256):
            self.texts = list(texts)
            self.labels = list(labels)
            self.tokenizer = tokenizer
            self.max_len = max_len
        def __len__(self):
            return len(self.texts)
        def __getitem__(self, idx):
            enc = self.tokenizer(str(self.texts[idx]), truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
            return {
                "input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels": torch.tensor(self.labels[idx], dtype=torch.long)
            }

    train_ds = TextClassificationDataset(train_df[TEXT_COL], train_df["label"], tokenizer, max_len=256)
    val_ds = TextClassificationDataset(val_df[TEXT_COL], val_df["label"], tokenizer, max_len=256)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, num_labels=num_classes, id2label=id2label, label2id=label2id)

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss_fct = torch.nn.CrossEntropyLoss(weight=weights_tensor)
            loss = loss_fct(outputs.get("logits").view(-1, self.model.config.num_labels), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    def compute_metrics(eval_pred):
        preds = np.argmax(eval_pred[0], axis=-1)
        return {
            "accuracy": accuracy_score(eval_pred[1], preds),
            "f1_macro": f1_score(eval_pred[1], preds, average="macro", zero_division=0)
        }

    training_args = TrainingArguments(
        output_dir="./checkpoints_multiclass",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        logging_steps=50,
        save_total_limit=1
    )

    trainer = WeightedTrainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
    print("Transformer siap dilatih (jalankan trainer.train() untuk mengeksekusi).")
except Exception as e:
    print("Catatan IndoBERT:", e)


## 5. Laporan Detail Kinerja per Kategori & Submission


In [ ]:
print("=== LAPORAN PERFORMA PER KATEGORI (BASELINE) ===")
print(classification_report(val_df["label"], lr_preds, target_names=[id2label[i] for i in sorted(val_df["label"].unique())], zero_division=0))

sub_df = pd.DataFrame({
    "id": val_df["id"] if "id" in val_df.columns else range(len(val_df)),
    "predicted_category": [id2label[p] for p in lr_preds]
})
sub_df.to_csv("submission_multiclass.csv", index=False)
print("Berkas submission_multiclass.csv berhasil disimpan.")
